<a href="https://colab.research.google.com/github/soyuz43/input-conditioned-gain/blob/main/Extract_v_from_Trivia_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U \
  "transformers>=4.39" \
  "datasets>=2.18" \
  "accelerate>=0.27" \
  bitsandbytes \
  safetensors

### Cell 0 — Constants, utilities, and hashing (no Drive writes)
Defines: canonical JSON hashing, timestamp formatting, atomic JSON write, and safe path helpers.
All later cells reuse these utilities so experiment identity and artifact naming are deterministic.

In [ ]:
# =========================
# Cell 0: Utilities
# =========================
import os, json, math, random, time, hashlib, datetime
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Optional

def utc_timestamp() -> str:
    # Filename-safe, lexicographically sortable, unambiguous UTC
    return datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")

def _json_default(o):
    # Avoid non-serializable objects leaking into config
    if hasattr(o, "tolist"):
        return o.tolist()
    if isinstance(o, (set,)):
        return sorted(list(o))
    raise TypeError(f"Object of type {type(o)} is not JSON serializable")

def canonical_json_dumps(obj: Any) -> str:
    # Canonical, stable hashing: sorted keys, no whitespace, stable separators
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), default=_json_default)

def sha256_hex(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def atomic_write_json(path: str, obj: Any) -> None:
    # Write to temp then replace to avoid partial files on crash
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, sort_keys=True, default=_json_default)
    os.replace(tmp, path)

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def file_exists(path: str) -> bool:
    return os.path.exists(path)

print("utils ready")

### Cell 1 — Deterministic Runtime Initialization (no Drive writes)

Establishes reproducibility controls and captures static runtime metadata.
Does not compute experiment hash.
Does not write artifacts.
Only prepares state for later env_snapshot capture.

In [ ]:
# =========================
# Cell 1: Deterministic Runtime Initialization
# =========================

import os, random, time
import numpy as np
import torch
import transformers
import datasets

# -------------------------
# Reproducibility
# -------------------------
SEED = 101

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_grad_enabled(False)

# Optional stricter determinism (safe for inference-only workloads)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# -------------------------
# Runtime Introspection
# -------------------------
cuda_available = torch.cuda.is_available()
cuda_version = torch.version.cuda
gpu_name = torch.cuda.get_device_name(0) if cuda_available else None

print("torch:", torch.__version__)
print("cuda available:", cuda_available)
print("cuda version:", cuda_version)
print("gpu:", gpu_name)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)

# -------------------------
# Environment Snapshot (in-memory only)
# -------------------------
env_snapshot = {
    "torch_version": torch.__version__,
    "cuda_available": cuda_available,
    "cuda_version": cuda_version,
    "gpu_name": gpu_name,
    "transformers_version": transformers.__version__,
    "datasets_version": datasets.__version__,
    "seed": SEED,
    "timestamp_session_start": int(time.time()),
}

print("env snapshot initialized (not yet saved)")

### Cell 2 — Drive Mount + Model Loader (no Drive writes)

Mounts Google Drive (for later atomic save only), defines EXPERIMENT_ROOT, logs into Hugging Face (if token present),
loads tokenizer + model (4-bit), and builds an in-memory `model_meta` dictionary used later in EXPERIMENT_CONFIG hashing.

In [ ]:
# =========================
# Cell 2: Drive + Model Loader
# =========================

from google.colab import drive
drive.mount("/content/drive")

EXPERIMENT_ROOT = "/content/drive/MyDrive/steering_experiments"
os.makedirs(EXPERIMENT_ROOT, exist_ok=True)
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)

from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in with HF_TOKEN from Colab Secrets.")
else:
    print("No HF_TOKEN found in Colab Secrets (public models will still work).")

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# =========================
# Model Selection
# =========================

MODEL_NAME = "Qwen/Qwen2.5-7B"
MAX_LENGTH = 512

# bitsandbytes int8 path uses fp16 in MatMul8bitLt for activations in practice,
# so make it explicit to avoid “bf16 requested, fp16 executed” ambiguity.
ACTIVATION_DTYPE = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=ACTIVATION_DTYPE,   # key change vs your prior 8-bit run
)
model.eval()

cfg = model.config
num_layers = getattr(cfg, "num_hidden_layers", None) or getattr(cfg, "n_layer", None)
hidden_size = getattr(cfg, "hidden_size", None) or getattr(cfg, "n_embd", None)

print("MODEL_NAME:", MODEL_NAME)
print("num_layers:", num_layers, "hidden_size:", hidden_size)

model_meta = {
    "model_name": MODEL_NAME,
    "num_layers": int(num_layers),
    "hidden_size": int(hidden_size),
    "max_length": int(MAX_LENGTH),
    "tokenizer": {
        "use_fast": True,
        "pad_token_id": int(tokenizer.pad_token_id),
        "eos_token_id": int(tokenizer.eos_token_id),
    },
    "quantization": {
        "load_in_8bit": True,
        "llm_int8_threshold": 6.0,
        "llm_int8_has_fp16_weight": False,

        # provenance fields that should participate in exp_hash
        "activation_input_dtype": str(ACTIVATION_DTYPE).replace("torch.", ""),
        "activation_compute_kernel": "bitsandbytes.MatMul8bitLt",
        "activation_compute_dtype_expected": "float16",
    },
}

print("model loader complete (no files written)")

### Cell 3 — Dataset Slice Module (first-class, hashable, no Drive writes)

Builds a deterministic TriviaQA slice.
Outputs:
  - train_qs
  - eval_qs
  - slice_meta  (fully hashable, no prompt strings stored)

This metadata becomes part of EXPERIMENT_CONFIG and therefore determines exp_hash.

In [ ]:
# =========================
# Cell 3: Dataset Slice
# =========================

from datasets import load_dataset

DATASET_NAME = "trivia_qa"
DATASET_CONFIG = "unfiltered.nocontext"

N_TOTAL = 1200
N_TRAIN = 800
N_EVAL  = 400
SHUFFLE_SEED = 42

MIN_Q_LEN = 10
MAX_Q_LEN = 200
REQUIRE_QUESTION_MARK = True

print("Loading dataset...")
ds = load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")

def valid_question(ex):
    q = ex.get("question", "").strip()
    if len(q) < MIN_Q_LEN:
        return None
    if len(q) > MAX_Q_LEN:
        return None
    if REQUIRE_QUESTION_MARK and not q.endswith("?"):
        return None
    return q

# Deterministic shuffle
all_indices = list(range(len(ds)))
rng = random.Random(SHUFFLE_SEED)
rng.shuffle(all_indices)

selected_indices = []
questions = []

for idx in all_indices:
    q = valid_question(ds[idx])
    if q is not None:
        selected_indices.append(idx)
        questions.append(q)
    if len(questions) >= N_TOTAL:
        break

assert len(questions) >= N_TOTAL, "Not enough valid questions"

train_qs = questions[:N_TRAIN]
eval_qs  = questions[N_TRAIN:N_TRAIN + N_EVAL]

# ---- Hashes (indices + question content) ----
indices_hash = sha256_hex(canonical_json_dumps(selected_indices[:N_TOTAL]))
question_hash = sha256_hex(canonical_json_dumps(questions[:N_TOTAL]))

slice_meta = {
    "dataset_name": DATASET_NAME,
    "dataset_config": DATASET_CONFIG,
    "n_total": N_TOTAL,
    "n_train": N_TRAIN,
    "n_eval": N_EVAL,
    "shuffle_seed": SHUFFLE_SEED,
    "filters": {
        "min_len": MIN_Q_LEN,
        "max_len": MAX_Q_LEN,
        "require_question_mark": REQUIRE_QUESTION_MARK,
    },
    "selected_indices": selected_indices[:N_TOTAL],
    "slice_hash": indices_hash,
    "question_hash": question_hash,
}

# Basic stats (not part of identity; diagnostic only)
avg_len = sum(len(q) for q in questions[:N_TOTAL]) / N_TOTAL
unique_ratio = len(set(questions[:N_TOTAL])) / N_TOTAL

print("Dataset slice ready")
print("Train:", len(train_qs), "Eval:", len(eval_qs))
print("Avg length:", round(avg_len, 1))
print("Uniqueness:", round(unique_ratio, 4))
print("Slice hash:", indices_hash[:12])

### Cell 4 — Structured Style Registry (explicit, hashable, no Drive writes)

Defines all style conditions as a first-class registry.
No string duplication inside the experiment logic.
Ordering variants are explicit.
No synonyms.
This registry becomes part of EXPERIMENT_CONFIG.

In [ ]:
# =========================
# Cell 4: Style Registry
# =========================

# Core semantic tokens we are testing
STYLE_TOKENS = {
    "concise_word": "concise",
    "verbose_word": "verbose",
    "concise_phrase": "Answer concisely.",
    "verbose_phrase": "Answer verbosely.",
}

BASE_ASSISTANT = "You are a helpful assistant."

style_registry = {
    # ─────────────────────────────────────────────
    # Canonical instruction forms
    # ─────────────────────────────────────────────
    "concise_standard": {
        "system_template": f"{BASE_ASSISTANT} {STYLE_TOKENS['concise_phrase']}",
        "family": "canonical",
        "polarity": "concise",
    },
    "verbose_standard": {
        "system_template": f"{BASE_ASSISTANT} {STYLE_TOKENS['verbose_phrase']}",
        "family": "canonical",
        "polarity": "verbose",
    },

    # ─────────────────────────────────────────────
    # Single-word variants appended
    # ─────────────────────────────────────────────
    "concise_word_appended": {
        "system_template": f"{BASE_ASSISTANT} {STYLE_TOKENS['concise_word']}",
        "family": "word_appended",
        "polarity": "concise",
    },
    "verbose_word_appended": {
        "system_template": f"{BASE_ASSISTANT} {STYLE_TOKENS['verbose_word']}",
        "family": "word_appended",
        "polarity": "verbose",
    },

    # ─────────────────────────────────────────────
    # Phrase before base role
    # ─────────────────────────────────────────────
    "concise_phrase_prepend": {
        "system_template": f"{STYLE_TOKENS['concise_phrase']} {BASE_ASSISTANT}",
        "family": "phrase_prepend",
        "polarity": "concise",
    },
    "verbose_phrase_prepend": {
        "system_template": f"{STYLE_TOKENS['verbose_phrase']} {BASE_ASSISTANT}",
        "family": "phrase_prepend",
        "polarity": "verbose",
    },

    # ─────────────────────────────────────────────
    # Word before base role
    # ─────────────────────────────────────────────
    "concise_word_prepend": {
        "system_template": f"{STYLE_TOKENS['concise_word']} {BASE_ASSISTANT}",
        "family": "word_prepend",
        "polarity": "concise",
    },
    "verbose_word_prepend": {
        "system_template": f"{STYLE_TOKENS['verbose_word']} {BASE_ASSISTANT}",
        "family": "word_prepend",
        "polarity": "verbose",
    },
}

print("Defined style conditions:")
for k in style_registry:
    print(" -", k)

print("Total style variants:", len(style_registry))

### Cell 5 — Prompt Builder + Representation Spec (no Drive writes)

Defines:
- Deterministic `make_chat_input(question, style_key)`
- Candidate layer selection
- Representation spec (anchor + layers)

No string duplication.
No persistence.
All fields become part of EXPERIMENT_CONFIG.

In [ ]:
# =========================
# Cell 5: Prompt Builder + Representation Spec
# =========================

# ---- Prompt Builder ----
def make_chat_input(question: str, style_key: str) -> str:
    if style_key not in style_registry:
        raise ValueError(f"Unknown style_key: {style_key}")

    system_msg = style_registry[style_key]["system_template"]

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": question},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


# ---- Candidate Layer Selection ----
def candidate_layers(num_layers: int, n_samples: int = 10):
    start = max(2, num_layers // 6)
    end   = max(start + 1, num_layers - 2)
    layers = list(sorted(set(
        np.linspace(start, end, n_samples).round().astype(int).tolist()
    )))
    return layers

LAYERS = candidate_layers(model_meta["num_layers"], n_samples=10)


# ---- Representation Spec ----
representation_spec = {
    "token_anchor": "last_prompt_token",
    "layers": LAYERS,
}


# ---- Sanity Print ----
print("Sample prompt preview:")
sample_style = list(style_registry.keys())[0]
print("Style key:", sample_style)
print(make_chat_input(train_qs[0], sample_style)[:300])
print()

print("Representation spec preview:")
print(json.dumps(representation_spec, indent=2))

### Cell 6 — Experiment Config Assembly + Deterministic Directory

Assembles EXPERIMENT_CONFIG.
Computes exp_hash from canonical JSON.
Resolves exp_dir = EXPERIMENT_ROOT/<exp_hash>.

Writes config.json only if it does not already exist.
Does not write run artifacts.

In [ ]:
# =========================
# Cell 6: Experiment Config + Directory (ENFORCED naming scheme)
# =========================

import os, re

# ---- Metrics + Statistical Spec ----
stats_spec = {
    "mean_estimator": "online_mean_float64",
    "batch_size": 8,
}

metrics_spec = {
    "primary_metric": "AUC",
    "secondary_metrics": ["mean_gap", "vnorm"],
}

# ---- Assemble EXPERIMENT_CONFIG (canonical identity inputs) ----
EXPERIMENT_CONFIG = {
    "dataset": slice_meta,
    "model": model_meta,
    "representation": representation_spec,
    "styles": style_registry,
    "stats": stats_spec,
    "metrics": metrics_spec,
}

# ---- Enforced naming scheme helpers ----
# Strict format: W<weight_precision>_<quant_scheme>_A<activation_precision>
# e.g. W8_INT8_A16FP, W4_NF4_A16BF, W16_FP16_A16FP, W16_BF16_A16BF
RE_LABEL = re.compile(r"^(W4|W8|W16)_(NF4|INT8|FP16|BF16)_A(16BF|16FP|32FP)$")

def infer_precision_components_from_model_meta(mm: dict) -> dict:
    """
    Infers {weight_precision, quant_scheme, activation_precision} from your model_meta,
    based on the conventions you've already started using.

    Required:
      mm["quantization"]["load_in_8bit"] OR mm["quantization"]["load_in_4bit"] OR neither
      mm["quantization"]["activation_compute_dtype_expected"] in {"float16","bfloat16","float32"} (optional but recommended)

    Returns:
      {
        "weight_precision": "W4"|"W8"|"W16",
        "quant_scheme": "NF4"|"INT8"|"FP16"|"BF16",
        "activation_precision": "16FP"|"16BF"|"32FP",
      }
    """
    q = (mm or {}).get("quantization", {}) or {}

    # ---- weights + quant scheme ----
    if bool(q.get("load_in_4bit", False)):
        weight_precision = "W4"
        quant_scheme = "NF4"  # bitsandbytes 4-bit path you described
    elif bool(q.get("load_in_8bit", False)):
        weight_precision = "W8"
        quant_scheme = "INT8"
    else:
        # Unquantized
        weight_precision = "W16"
        # Prefer explicit dtype from model_meta if you store it; otherwise default to FP16
        # (You can add mm["full_precision_dtype"] = "fp16"/"bf16" upstream if you want.)
        full = (mm or {}).get("full_precision_dtype", None)
        if full is not None:
            full = str(full).lower()
        if full in ("bf16", "bfloat16"):
            quant_scheme = "BF16"
        else:
            quant_scheme = "FP16"

    # ---- activation precision ----
    act = str(q.get("activation_compute_dtype_expected", "")).lower().strip()
    if act in ("float16", "fp16"):
        activation_precision = "16FP"
    elif act in ("bfloat16", "bf16"):
        activation_precision = "16BF"
    elif act in ("float32", "fp32"):
        activation_precision = "32FP"
    else:
        # Fallback: infer from any torch dtype string you stored
        act_in = str(q.get("activation_input_dtype", "")).lower()
        if "float16" in act_in or "fp16" in act_in:
            activation_precision = "16FP"
        elif "bfloat16" in act_in or "bf16" in act_in:
            activation_precision = "16BF"
        elif "float32" in act_in or "fp32" in act_in:
            activation_precision = "32FP"
        else:
            # Last-resort default: fp16 (most common in Colab paths)
            activation_precision = "16FP"

    return {
        "weight_precision": weight_precision,
        "quant_scheme": quant_scheme,
        "activation_precision": activation_precision,
    }

def build_precision_label(mm: dict) -> str:
    c = infer_precision_components_from_model_meta(mm)
    label = f"{c['weight_precision']}_{c['quant_scheme']}_A{c['activation_precision']}"
    if not RE_LABEL.match(label):
        raise ValueError(f"PRECISION_LABEL failed validation: {label}")
    return label

# ---- Build + record label (human-readable wrapper) ----
PRECISION_LABEL = build_precision_label(model_meta)

# Record the label + components inside the config so provenance survives moves/copies
EXPERIMENT_CONFIG["run_label"] = PRECISION_LABEL
EXPERIMENT_CONFIG["precision_components"] = infer_precision_components_from_model_meta(model_meta)

# ---- Compute Deterministic Hash (canonical identity) ----
# IMPORTANT: hash AFTER run_label is inserted so the label is part of provenance.
config_json_canonical = canonical_json_dumps(EXPERIMENT_CONFIG)
exp_hash = sha256_hex(config_json_canonical)[:12]

# ---- Directory layout ----
# steering_experiments/<PRECISION_LABEL>/<exp_hash>/
exp_dir = os.path.join(EXPERIMENT_ROOT, PRECISION_LABEL, exp_hash)
ensure_dir(exp_dir)

print("Experiment label:", PRECISION_LABEL)
print("Experiment hash:", exp_hash)
print("Experiment directory:", exp_dir)

# ---- Write config.json only once ----
config_path = os.path.join(exp_dir, "config.json")
if not file_exists(config_path):
    atomic_write_json(config_path, EXPERIMENT_CONFIG)
    print("config.json written (new experiment)")
else:
    print("config.json already exists (identical config)")

print("Config assembly complete.")

### Cell 7 — Core Steering Computation (no Drive writes)

Computes:
- Mean representations per style
- v* vectors (verbose − concise) per family
- Norms
- Best layer by norm

No artifacts saved.
Pure in-memory execution.

In [ ]:
# =========================
# Cell 7: Core Steering Computation
# =========================

BATCH_SIZE = stats_spec["batch_size"]

def last_token_index(attn_mask: torch.Tensor):
    return attn_mask.long().sum(dim=1) - 1


@torch.no_grad()
def extract_last_prompt_vectors(text_batch, layers):
    tok = tokenizer(
        text_batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=model_meta["max_length"],
    )

    device = next(iter(model.parameters())).device
    tok = {k: v.to(device) for k, v in tok.items()}

    out = model(**tok, output_hidden_states=True, use_cache=False, return_dict=True)
    hs = out.hidden_states

    idx = last_token_index(tok["attention_mask"])
    B = idx.shape[0]

    results = {}
    for L in layers:
        hL = hs[L+1]
        v  = hL[torch.arange(B), idx, :]
        results[L] = v.float().cpu()
    return results


# ---- Initialize Means Per Style ----
def init_means():
    return {
        style_key: {L: torch.zeros(hidden_size, dtype=torch.float64) for L in LAYERS}
        for style_key in style_registry
    }, {style_key: 0 for style_key in style_registry}


def update_means(mean_dict, count_dict, batch_vectors):
    for style_key, layer_dict in batch_vectors.items():
        B = next(iter(layer_dict.values())).shape[0]
        new_n = count_dict[style_key] + B

        for L, X in layer_dict.items():
            s = X.double().sum(dim=0)
            mean_dict[style_key][L] = (
                mean_dict[style_key][L]
                + (s - B * mean_dict[style_key][L]) / new_n
            )

        count_dict[style_key] = new_n

    return mean_dict, count_dict


mean_dict, count_dict = init_means()

print("Starting mean accumulation...")

for start in range(0, len(train_qs), BATCH_SIZE):
    qs = train_qs[start:start+BATCH_SIZE]

    batch_vectors = {}

    for style_key in style_registry:
        prompts = [make_chat_input(q, style_key) for q in qs]
        batch_vectors[style_key] = extract_last_prompt_vectors(prompts, LAYERS)

    mean_dict, count_dict = update_means(mean_dict, count_dict, batch_vectors)

    if (start // BATCH_SIZE) % 20 == 0:
        print(f"Processed {min(start+BATCH_SIZE, len(train_qs))}/{len(train_qs)}")


print("Mean accumulation complete.")


# ---- Construct v* per family (verbose - concise) ----
vstars = {}
norms = {}

for style_key, spec in style_registry.items():
    if spec["polarity"] != "verbose":
        continue

    # Find matching concise partner in same family
    family = spec["family"]
    concise_partner = None
    for k, v in style_registry.items():
        if v["family"] == family and v["polarity"] == "concise":
            concise_partner = k
            break

    if concise_partner is None:
        continue

    family_name = family
    vstars[family_name] = {}
    norms[family_name] = {}

    for L in LAYERS:
        v = (mean_dict[style_key][L] - mean_dict[concise_partner][L]).float()
        vstars[family_name][L] = v
        norms[family_name][L] = torch.norm(v).item()


# ---- Best Layer by Norm Per Family ----
best_layers_by_norm = {}

for family in norms:
    best_L = max(norms[family], key=lambda L: norms[family][L])
    best_layers_by_norm[family] = best_L

print("Best layers by norm:")
for family, L in best_layers_by_norm.items():
    print(f"  {family}: layer {L} (||v*|| = {norms[family][L]:.4f})")

### Cell 8 — Evaluation (AUC + Mean Gap per Family, no Drive writes)

For each style family:
- Uses the learned v* (verbose − concise)
- Projects eval representations
- Computes AUC and mean gap per layer
- Selects best layer by AUC

No persistence. Pure evaluation phase.

In [ ]:
# =========================
# Cell 8: Evaluation Phase
# =========================

import numpy as np

def proj_scores_pair(questions, layer, v_unit, verbose_key, concise_key):
    scores_v = []
    scores_c = []

    device = next(model.parameters()).device
    v_unit = v_unit.to(device).float()

    for start in range(0, len(questions), BATCH_SIZE):
        qs = questions[start:start+BATCH_SIZE]

        verb_prompts = [make_chat_input(q, verbose_key) for q in qs]
        conc_prompts = [make_chat_input(q, concise_key) for q in qs]

        bv = extract_last_prompt_vectors(verb_prompts, [layer])[layer]
        bc = extract_last_prompt_vectors(conc_prompts, [layer])[layer]

        bv = bv.to(device).float()
        bc = bc.to(device).float()

        sv = (bv @ v_unit).cpu().numpy()
        sc = (bc @ v_unit).cpu().numpy()

        scores_v.append(sv)
        scores_c.append(sc)

    return np.concatenate(scores_v), np.concatenate(scores_c)


def auc_from_scores(pos, neg):
    cnt = 0.0
    tot = 0.0
    for p in pos:
        cnt += (p > neg).sum() + 0.5 * (p == neg).sum()
        tot += len(neg)
    return float(cnt / tot)


eval_report = {}
best_layers_by_auc = {}

print("Starting evaluation...")

for family in vstars:
    eval_report[family] = {}
    family_layers = vstars[family]

    # find keys
    verbose_key = None
    concise_key = None
    for k, v in style_registry.items():
        if v["family"] == family:
            if v["polarity"] == "verbose":
                verbose_key = k
            elif v["polarity"] == "concise":
                concise_key = k

    for L, v in family_layers.items():
        if torch.norm(v) < 1e-8:
            continue

        v_unit = (v / torch.norm(v)).cpu()

        sv, sc = proj_scores_pair(
            eval_qs,
            L,
            v_unit,
            verbose_key,
            concise_key
        )

        auc = auc_from_scores(sv, sc)
        gap = float(np.mean(sv) - np.mean(sc))

        eval_report[family][L] = {
            "auc": auc,
            "mean_gap": gap,
            "vnorm": norms[family][L],
        }

        print(f"[{family}] Layer {L:2d} | AUC={auc:.3f} | gap={gap:.4f}")

    # best by AUC
    best_L = max(eval_report[family], key=lambda L: eval_report[family][L]["auc"])
    best_layers_by_auc[family] = best_L

print("\nBest layers by AUC:")
for family, L in best_layers_by_auc.items():
    print(f"  {family}: layer {L} (AUC={eval_report[family][L]['auc']:.3f})")

### Cell 9 — Atomic Save Phase (Drive writes happen only here)

Writes (timestamped, never overwritten) into exp_dir:
- artifact_<ts>.pt      (tensors + full run payload)
- summary_<ts>.json     (no tensors)
- env_<ts>.json         (runtime snapshot)

Updates exp_dir/run_index.json by appending a new entry.
No other cells write artifacts.

In [ ]:
# =========================
# Cell 9: Atomic Save Phase
# =========================

import torch

run_ts = utc_timestamp()
run_id = run_ts  # keep identical; ledger-friendly

artifact_file = f"artifact_{run_ts}.pt"
summary_file  = f"summary_{run_ts}.json"
env_file      = f"env_{run_ts}.json"

artifact_path = os.path.join(exp_dir, artifact_file)
summary_path  = os.path.join(exp_dir, summary_file)
env_path      = os.path.join(exp_dir, env_file)

# ---- Extend env snapshot with run-local info (still just data) ----
env_snapshot_run = dict(env_snapshot)
env_snapshot_run.update({
    "run_id": run_id,
    "timestamp_utc": run_ts,
    "model_name": model_meta["model_name"],
})

# ---- Build artifact (with tensors) ----
artifact = {
    "run_id": run_id,
    "timestamp_utc": run_ts,
    "exp_hash": exp_hash,
    "experiment_config_sha256": sha256_hex(canonical_json_dumps(EXPERIMENT_CONFIG)),
    "model_meta": model_meta,
    "dataset_meta": slice_meta,
    "representation_spec": representation_spec,
    "styles": style_registry,

    # results
    "layers": LAYERS,
    "best_layers_by_norm": {k: int(v) for k, v in best_layers_by_norm.items()},
    "best_layers_by_auc": {k: int(v) for k, v in best_layers_by_auc.items()},
    "eval_report": eval_report,
    "norms": norms,

    # tensors
    "v_star_raw": {
        family: {int(L): vstars[family][L].cpu() for L in vstars[family]}
        for family in vstars
    },
    "v_star_unit": {
        family: {
            int(L): (vstars[family][L] / (torch.norm(vstars[family][L]) + 1e-12)).cpu()
            for L in vstars[family]
        }
        for family in vstars
    },
}

# ---- Summary (no tensors) ----
summary = dict(artifact)
summary["v_star_raw"] = {
    family: {int(L): "tensor[d]" for L in artifact["v_star_raw"][family]}
    for family in artifact["v_star_raw"]
}
summary["v_star_unit"] = {
    family: {int(L): "tensor[d]" for L in artifact["v_star_unit"][family]}
    for family in artifact["v_star_unit"]
}

# ---- Write files (timestamped; should not already exist) ----
assert not file_exists(artifact_path), f"Refusing overwrite: {artifact_path}"
assert not file_exists(summary_path),  f"Refusing overwrite: {summary_path}"
assert not file_exists(env_path),      f"Refusing overwrite: {env_path}"

torch.save(artifact, artifact_path)
atomic_write_json(summary_path, summary)
atomic_write_json(env_path, env_snapshot_run)

print("Saved:")
print(" -", artifact_path)
print(" -", summary_path)
print(" -", env_path)

# ---- Update run_index.json (append-only ledger) ----
run_index_path = os.path.join(exp_dir, "run_index.json")

if file_exists(run_index_path):
    with open(run_index_path, "r") as f:
        run_index = json.load(f)
        if not isinstance(run_index, list):
            raise ValueError("run_index.json exists but is not a list")
else:
    run_index = []

# Small per-family max AUC summary
max_auc_by_family = {
    family: float(eval_report[family][best_layers_by_auc[family]]["auc"])
    for family in best_layers_by_auc
}

run_index.append({
    "run_id": run_id,
    "timestamp_utc": run_ts,
    "artifact_file": artifact_file,
    "summary_file": summary_file,
    "env_file": env_file,
    "best_layers_by_norm": {k: int(v) for k, v in best_layers_by_norm.items()},
    "best_layers_by_auc": {k: int(v) for k, v in best_layers_by_auc.items()},
    "max_auc_by_family": max_auc_by_family,
})

atomic_write_json(run_index_path, run_index)

print("Updated ledger:", run_index_path)
print("Total runs in this experiment:", len(run_index))

### Cell 10 — Rotation Tests (scoped subdirectory + ledger linkage)

Creates a rotation namespace under:
  exp_dir/rotations/<rot_hash>/

Writes (immutable):
- rotation_config.json (only if absent)
- rotation_results_<ts>.json (timestamped)

Also appends a rotation summary object into exp_dir/run_index.json
under the most recent run entry (matching run_id).

# Run the rotation ensemble test



In [ ]:
# =========================
# Cell 10: Rotation Tests (scoped + versioned)
# =========================

import numpy as np
import torch

# ---- Rotation configuration (THIS determines rot_hash) ----
rotation_config = {
    "method": "haar_like_qr",         # QR on Gaussian matrix
    "n_rot": 10,
    "n_samples": 100,
    "layers_to_test": "best_by_auc_per_family",  # interpreted below
    "dtype": "float32",
}

rot_hash = sha256_hex(canonical_json_dumps(rotation_config))[:8]
rot_dir = os.path.join(exp_dir, "rotations", rot_hash)
ensure_dir(rot_dir)

rot_cfg_path = os.path.join(rot_dir, "rotation_config.json")
if not file_exists(rot_cfg_path):
    atomic_write_json(rot_cfg_path, rotation_config)

rot_ts = utc_timestamp()
rot_results_file = f"rotation_results_{rot_ts}.json"
rot_results_path = os.path.join(rot_dir, rot_results_file)

assert not file_exists(rot_results_path), f"Refusing overwrite: {rot_results_path}"

# ---- Helpers ----
def auc_from_scores(pos, neg):
    cnt = 0.0
    tot = 0.0
    for p in pos:
        cnt += (p > neg).sum() + 0.5 * (p == neg).sum()
        tot += len(neg)
    return float(cnt / tot)

def proj_scores_pair(questions, layer, v_unit, verbose_key, concise_key):
    scores_v = []
    scores_c = []

    device = next(model.parameters()).device
    v_unit = v_unit.to(device).float()

    for start in range(0, len(questions), BATCH_SIZE):
        qs = questions[start:start+BATCH_SIZE]

        verb_prompts = [make_chat_input(q, verbose_key) for q in qs]
        conc_prompts = [make_chat_input(q, concise_key) for q in qs]

        bv = extract_last_prompt_vectors(verb_prompts, [layer])[layer].to(device).float()
        bc = extract_last_prompt_vectors(conc_prompts, [layer])[layer].to(device).float()

        sv = (bv @ v_unit).cpu().numpy()
        sc = (bc @ v_unit).cpu().numpy()

        scores_v.append(sv)
        scores_c.append(sc)

    return np.concatenate(scores_v), np.concatenate(scores_c)

def rotation_test_for_family(family: str, layer: int, v_unit_cpu: torch.Tensor, n_rot: int, n_samples: int):
    device = next(model.parameters()).device
    dim = int(v_unit_cpu.shape[0])

    # stable keys for prompts
    verbose_key = None
    concise_key = None
    for k, v in style_registry.items():
        if v["family"] == family:
            if v["polarity"] == "verbose":
                verbose_key = k
            elif v["polarity"] == "concise":
                concise_key = k
    if verbose_key is None or concise_key is None:
        raise ValueError(f"Missing style keys for family={family}")

    # move base vector to device once
    v_unit = v_unit_cpu.to(device).float()

    aucs = []
    for i in range(n_rot):
        # Random orthogonal matrix via QR on GPU
        rand_mat = torch.randn(dim, dim, device=device, dtype=torch.float32)
        R, _ = torch.linalg.qr(rand_mat)

        rot_v = (R @ v_unit).contiguous()

        sv, sc = proj_scores_pair(eval_qs[:n_samples], layer, rot_v, verbose_key, concise_key)
        aucs.append(auc_from_scores(sv, sc))

    return {
        "family": family,
        "layer": int(layer),
        "n_rot": int(n_rot),
        "n_samples": int(n_samples),
        "mean_auc": float(np.mean(aucs)) if len(aucs) else None,
        "std_auc": float(np.std(aucs)) if len(aucs) else None,
        "aucs": [float(x) for x in aucs],
    }

# ---- Decide which family/layer pairs to test ----
targets = []
for family, bestL in best_layers_by_auc.items():
    targets.append((family, int(bestL)))

# ---- Run rotation ensemble ----
rotation_results = {
    "rotation_hash": rot_hash,
    "rotation_timestamp_utc": rot_ts,
    "rotation_config": rotation_config,
    "run_id_link": run_id,
    "targets": [],
}

n_rot = int(rotation_config["n_rot"])
n_samples = int(rotation_config["n_samples"])

print("Running rotation tests:")
for (family, layer) in targets:
    v_unit_cpu = artifact["v_star_unit"][family][layer].clone()  # CPU tensor
    res = rotation_test_for_family(family, layer, v_unit_cpu, n_rot=n_rot, n_samples=n_samples)
    rotation_results["targets"].append(res)
    print(f"  [{family}] layer {layer} -> mean AUC {res['mean_auc']:.3f} ± {res['std_auc']:.3f}")

# ---- Save rotation results ----
atomic_write_json(rot_results_path, rotation_results)
print("Saved rotation results:", rot_results_path)

# ---- Append rotation summary into run_index.json for this run_id ----
run_index_path = os.path.join(exp_dir, "run_index.json")
with open(run_index_path, "r") as f:
    run_index = json.load(f)
if not isinstance(run_index, list):
    raise ValueError("run_index.json is not a list")

# Find the matching run entry (latest matching run_id)
match_i = None
for i in range(len(run_index)-1, -1, -1):
    if run_index[i].get("run_id") == run_id:
        match_i = i
        break
if match_i is None:
    raise ValueError(f"Could not find run_id={run_id} in run_index.json")

rot_summary = {
    "rotation_hash": rot_hash,
    "rotation_timestamp_utc": rot_ts,
    "rotation_dir": f"rotations/{rot_hash}",
    "rotation_results_file": rot_results_file,
    "by_family": {
        t["family"]: {
            "layer": t["layer"],
            "mean_auc": t["mean_auc"],
            "std_auc": t["std_auc"],
            "n_rot": t["n_rot"],
            "n_samples": t["n_samples"],
        }
        for t in rotation_results["targets"]
    }
}

if "rotations" not in run_index[match_i]:
    run_index[match_i]["rotations"] = []
run_index[match_i]["rotations"].append(rot_summary)

atomic_write_json(run_index_path, run_index)
print("Linked rotation summary into ledger entry for run_id:", run_id)

In [ ]:
# =========================
# Cell 10.5: Clear VRAM (minimal)
# =========================
import gc
import torch

# Drop large objects if they exist
for name in ["model", "gauge_model"]:
    if name in globals():
        del globals()[name]

# Also drop any leftover big tensors you might have kept around
for name in ["artifact", "loaded"]:
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Cleared VRAM (best-effort).")